# Hybrid Retrieval: BM25 + Dense (FAISS) with RRF

**Goal:** Add a lexical retrieval channel (BM25) on top of dense embeddings, then merge ranked lists with **Reciprocal Rank Fusion (RRF)**.

**Why this is a separate notebook:** it is a retrieval-system upgrade orthogonal to the text-ablation storyline in Notebook 3.

**Eval protocol:** item-to-item co-preference (same as Notebooks 2/3): query = one ordered dish; relevant = other orders/favorites for that user.


In [ ]:
import torch

assert torch.cuda.is_available(), "GPU recommended for encoding (enable GPU on Colab/Kaggle)."
print(f"✓ GPU: {torch.cuda.get_device_name(0)}")


In [ ]:
# ============================================================
# SETUP
# ============================================================
import os, sys

REPO = "Embedding-Based-Recommender"
GITHUB_USER = "IldarRakiev"

ENV = 'kaggle' if os.path.exists('/kaggle/working') else 'colab' if os.path.exists('/content') else 'local'
BASE = '/kaggle/working' if ENV == 'kaggle' else '/content' if ENV == 'colab' else os.getcwd()
REPO_DIR = f'{BASE}/{REPO}' if ENV != 'local' else os.path.abspath(os.path.join(os.getcwd(), '..'))

if ENV != 'local':
    if not os.path.exists(REPO_DIR):
        os.system(f'git clone https://github.com/{GITHUB_USER}/{REPO}.git {REPO_DIR}')
    else:
        os.system(f'cd {REPO_DIR} && git pull -q')

os.system('pip install -q sentence-transformers faiss-cpu pandas pyarrow matplotlib seaborn umap-learn plotly scikit-learn tqdm rank-bm25')
sys.path.insert(0, f'{REPO_DIR}/src')

print(f"Environment: {ENV} | Repo: {REPO_DIR}")
print("Setup complete")


In [ ]:
# ============================================================
# DATA PATHS
# ============================================================
import os

ENV = 'kaggle' if os.path.exists('/kaggle/working') else 'colab' if os.path.exists('/content') else 'local'

if ENV == 'local':
    SYNTHETIC_DIR = os.path.join(os.path.dirname(os.getcwd()), 'data', 'synthetic')
else:
    SYNTHETIC_DIR = os.path.join(REPO_DIR, 'data', 'synthetic')

OUTPUT_DIR = '/kaggle/working/processed' if ENV == 'kaggle' else SYNTHETIC_DIR
os.makedirs(OUTPUT_DIR, exist_ok=True)

print(f"SYNTHETIC_DIR = {SYNTHETIC_DIR}")
print(f"OUTPUT_DIR    = {OUTPUT_DIR}")


In [ ]:
import json
import faiss
import numpy as np
import pandas as pd

from text_builders import dish_to_rich_text
from embedding_model import EmbeddingModel
from hybrid_rrf import HybridRRFConfig, build_bm25_index, dish_text_for_bm25, hybrid_search_dish_ids
from utils import evaluate_all

np.random.seed(42)

dishes = pd.read_parquet(f"{SYNTHETIC_DIR}/dishes.parquet")
test = pd.read_parquet(f"{SYNTHETIC_DIR}/interactions_test.parquet")

dish_id_to_idx = {did: i for i, did in enumerate(dishes['id'])}
idx_to_dish_id = {i: did for i, did in enumerate(dishes['id'])}

# Same "best text" config used in Notebook 3's `full_improved` experiment
id_to_text = {
    row['id']: dish_to_rich_text(
        row.to_dict(),
        tags=row.get('tag_list', []),
        include_recipe=False,
        include_macro_tokens=False,
        include_ratios=True,
        include_ingredients=True,
    )
    for _, row in dishes.iterrows()
}

print(f"Dishes: {len(dishes):,} | Test interactions: {len(test):,}")


In [ ]:
model = EmbeddingModel()
print(f"Dense model: {model.model_name} | dim={model.dim}")

texts = [id_to_text[rid] for rid in dishes['id'] if rid in id_to_text]
embs = model.encode(texts, batch_size=64).astype(np.float32)

idx = faiss.IndexFlatIP(model.dim)
idx.add(embs)

bm25_docs = [dish_text_for_bm25(row.to_dict(), tags=row.get('tag_list', [])) for _, row in dishes.iterrows()]
bm25_index = build_bm25_index(bm25_docs)

hybrid_cfg = HybridRRFConfig(dense_k=200, bm25_k=200, rrf_k=60)
print("Indexes ready: FAISS + BM25")


In [ ]:
def evaluate_retrieval_dense(index, embeddings, ks=None):
    if ks is None:
        ks = [5, 10, 20]

    user_positives = (
        test[test['interaction_type'].isin(['order', 'favorite'])]
        .groupby('user_id')['dish_id']
        .apply(set)
        .to_dict()
    )

    rows = []
    for _, pos_dishes in user_positives.items():
        pos_dishes = {d for d in pos_dishes if d in dish_id_to_idx}
        if len(pos_dishes) < 5:
            continue

        query_dish = list(pos_dishes)[0]
        relevant = pos_dishes - {query_dish}

        q_idx = dish_id_to_idx[query_dish]
        _, inds = index.search(embeddings[q_idx:q_idx + 1], max(ks) + 1)
        recommended = [idx_to_dish_id[i] for i in inds[0] if i >= 0 and idx_to_dish_id.get(i) != query_dish]
        rows.append(evaluate_all(recommended, relevant, ks=ks))

    return pd.DataFrame(rows).mean().to_dict() if rows else {}


def evaluate_retrieval_hybrid_rrf(index, embeddings, ks=None):
    if ks is None:
        ks = [5, 10, 20]

    user_positives = (
        test[test['interaction_type'].isin(['order', 'favorite'])]
        .groupby('user_id')['dish_id']
        .apply(set)
        .to_dict()
    )

    rows = []
    for _, pos_dishes in user_positives.items():
        pos_dishes = {d for d in pos_dishes if d in dish_id_to_idx}
        if len(pos_dishes) < 5:
            continue

        query_dish = list(pos_dishes)[0]
        relevant = pos_dishes - {query_dish}

        q_idx = dish_id_to_idx[query_dish]
        q_text = id_to_text.get(query_dish, '')

        ranked = hybrid_search_dish_ids(
            query_text=q_text,
            query_vec=embeddings[q_idx],
            bm25_index=bm25_index,
            faiss_index=index,
            cfg=hybrid_cfg,
            final_top_k=max(ks) + 1,
        )

        recommended = [idx_to_dish_id[i] for i, _ in ranked if idx_to_dish_id.get(i) != query_dish]
        rows.append(evaluate_all(recommended, relevant, ks=ks))

    return pd.DataFrame(rows).mean().to_dict() if rows else {}


metrics_dense = evaluate_retrieval_dense(idx, embs)
metrics_hybrid = evaluate_retrieval_hybrid_rrf(idx, embs)

comparison = pd.DataFrame({"dense_only": metrics_dense, "bm25_dense_rrf": metrics_hybrid}).T
print(comparison[["P@5", "P@10", "NDCG@10", "MRR"]].round(4))

delta_p10 = metrics_hybrid.get("P@10", 0) - metrics_dense.get("P@10", 0)
print(f"\nΔ P@10 (hybrid - dense): {delta_p10:+.4f}")


In [ ]:
out = {"dense_only": metrics_dense, "bm25_dense_rrf": metrics_hybrid}

with open(os.path.join(OUTPUT_DIR, "results_hybrid_bm25_dense_rrf.json"), "w", encoding="utf-8") as f:
    json.dump(out, f, indent=2)

print(f"Saved: {os.path.join(OUTPUT_DIR, 'results_hybrid_bm25_dense_rrf.json')}")
